
# COVID-19 Time Series Forecasting using ARIMA  
### Dataset: Global COVID-19 Daily Confirmed Cases (Kaggle)

**Dataset link (DOWNLOAD MANUALLY):**  
https://www.kaggle.com/datasets/imdevskp/corona-virus-report

### Goal
Forecast **daily confirmed COVID-19 cases** for a single country using an **ARIMA model** and understand **why ARIMA has limitations for pandemic data**.



## Step 1: Import Required Libraries

We import:
- `pandas` for data manipulation  
- `matplotlib` for visualization  
- `statsmodels` for time series analysis (ADF test, ACF, PACF, ARIMA)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA



## Step 2: Load the Dataset

After downloading from Kaggle, place the file  
**`covid_19_clean_complete.csv`**  
in the same directory as this notebook.


In [ ]:

df = pd.read_csv("covid_19_clean_complete.csv")
df.head()



## Step 3: Filter Data for One Country (India)

ARIMA is a **univariate model**, so we:
- Filter for a single country
- Use daily confirmed cases


In [ ]:

india_df = df[df['Country/Region'] == 'India']
india_df = india_df[['Date', 'Confirmed']]

india_df['Date'] = pd.to_datetime(india_df['Date'])
india_df.set_index('Date', inplace=True)

india_df.head()



## Step 4: Visualize the Time Series

This plot helps us identify:
- Trend
- Spikes
- Non-stationarity


In [ ]:

plt.figure(figsize=(10,5))
plt.plot(india_df['Confirmed'])
plt.title("Daily Confirmed COVID-19 Cases - India")
plt.xlabel("Date")
plt.ylabel("Cases")
plt.show()



### Observation

- Strong upward and downward trends  
- Sudden spikes  
- Data is **non-stationary**



## Step 5: Stationarity Check (ADF Test)

ARIMA **requires stationary data**.


In [ ]:

adf_result = adfuller(india_df['Confirmed'])
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])



### Interpretation

- p-value > 0.05 → **Not stationary**
- We must difference the data



## Step 6: Differencing to Make Data Stationary

Differencing removes trend.


In [ ]:

diff_series = india_df['Confirmed'].diff().dropna()

plt.figure(figsize=(10,5))
plt.plot(diff_series)
plt.title("Differenced Series")
plt.show()



## Step 7: Re-check Stationarity


In [ ]:

adfuller(diff_series)



- p-value < 0.05  
✔ Data is now stationary  
📌 Therefore, **d = 1**



## Step 8: Identify p and q using ACF & PACF


In [ ]:

plot_acf(diff_series, lags=20)
plt.show()

plot_pacf(diff_series, lags=20)
plt.show()



### Interpretation

- PACF cuts off early → AR term (p)
- ACF cuts off early → MA term (q)

We choose:
📌 **ARIMA(1,1,1)**



## Step 9: Train ARIMA Model


In [ ]:

model = ARIMA(india_df['Confirmed'], order=(1,1,1))
model_fit = model.fit()

print(model_fit.summary())



## Step 10: Model Diagnostics

Residuals should:
- Have zero mean
- Show no autocorrelation
- Look like white noise


In [ ]:

model_fit.plot_diagnostics(figsize=(12,8))
plt.show()



## Step 11: Forecast Future Cases (Next 14 Days)


In [ ]:

forecast = model_fit.forecast(steps=14)
forecast



## ⚠️ VERY IMPORTANT: Limitations of ARIMA for COVID-19

### ARIMA Assumptions:
- Patterns change slowly
- No sudden external shocks

### COVID-19 Reality:
- Lockdowns
- Policy changes
- Testing changes
- Vaccination drives
- New variants

### Result:
👉 These cause **STRUCTURAL BREAKS**

ARIMA **cannot model regime changes**, so:
- Short-term forecasts only
- Long-term forecasts are unreliable



## Interview-Ready Conclusion

> “ARIMA can be applied to COVID-19 daily cases for short-term forecasting after differencing, but due to frequent structural breaks caused by policy interventions and variants, it should only be used as a baseline model.”
